In [7]:
import os
import random
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/get_plaintiffs_defendants.csv")

# ────────────────────────────────────────────────
# 3. Assign annotators randomly (Brian, Parker, Victor)
# ────────────────────────────────────────────────
cases = list(data["CASE_NUMBER"].unique())
random.seed(42)
random.shuffle(cases)
n = len(cases)

part1, part2, part3 = cases[: n // 3], cases[n // 3 : 2 * n // 3], cases[2 * n // 3 :]

def assign_annotator(case_number):
    if case_number in part1:
        return "Brian"
    elif case_number in part2:
        return "Parker"
    else:
        return "Victor"

data["annotator"] = data["CASE_NUMBER"].apply(assign_annotator)

# ────────────────────────────────────────────────
# 4. Clean up text fields (avoid NaN/float errors)
# ────────────────────────────────────────────────
data["FULL_NAME"] = data["FULL_NAME"].fillna("").astype(str)
data["PERSON_ROLE"] = data["PERSON_ROLE"].fillna("").astype(str)

# ────────────────────────────────────────────────
# 5. Aggregate one row per case
# ────────────────────────────────────────────────
grouped = (
    data.groupby("CASE_NUMBER")
    .apply(lambda g: pd.Series({
        "case_number": g.name,
        "case_hashkey": g["CASE_HASHKEY"].iloc[0] if "CASE_HASHKEY" in g else None,
        "complaint_filed_date": g["COMPLAINT_FILED_DATE"].iloc[0] if "COMPLAINT_FILED_DATE" in g else None,
        "plaintiff": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "plaintiff", "FULL_NAME"] if pd.notna(x) and x],
        "defendant": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "defendant", "FULL_NAME"] if pd.notna(x) and x],
        "annotator": g["annotator"].iloc[0],
    }))
    .reset_index(drop=True)
)

# ────────────────────────────────────────────────
# 6. Upload to Supabase (batch, fast)
# ────────────────────────────────────────────────
records = grouped.to_dict(orient="records")

response = supabase.table("cases_gold").upsert(records).execute()

print("✅ Upload complete!")
print("Inserted/updated rows:", len(records))
print("\nAnnotator distribution:")
print(grouped["annotator"].value_counts())

# Optional: preview one example
print("\nExample row:")
print(grouped.head(1).to_dict(orient="records")[0])


/var/folders/k7/b0_b7t6j6n72t68sh4s7t8400000gn/T/ipykernel_16053/3045081366.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


✅ Upload complete!
Inserted/updated rows: 100

Annotator distribution:
annotator
Victor    34
Brian     33
Parker    33
Name: count, dtype: int64

Example row:
{'case_number': '24CHLC00247', 'case_hashkey': 413524874617169216, 'complaint_filed_date': '2024-01-04 00:00:00.000', 'plaintiff': ['Midland Credit Management Inc'], 'defendant': ['MARCO A CABRERA, an Individual'], 'annotator': 'Victor'}
